# Cipher Code Kraken: Training Data Curation Pipeline

This notebook orchestrates the full data curation pipeline for the Cipher Code Kraken training.
It collects creative code from GitHub repos and CodePen pens, formats them into 4 training datasets:

1. **SFT** (Supervised Fine-Tuning): Chat-template instruction-response pairs (~2000+ examples)
2. **SimPO** (Simple Preference Optimization): Chosen/rejected pairs (~1000+ pairs)
3. **GRPO** (Group Relative Policy Optimization): Prompt-only dataset (~500+ prompts)
4. **KTO** (Kahneman-Tversky Optimization): Binary-label triples (~500+ entries)

**Anti-Slop Philosophy:** All data is filtered through the slop detector. Creative code with Three.js,
GSAP, Lenis, WebGL shaders, and advanced CSS is prioritized. Generic template garbage is rejected.

---

In [ ]:
# Cell 1: Install dependencies
# PyGithub for GitHub API, requests + beautifulsoup4 for CodePen scraping
!pip install PyGithub requests beautifulsoup4 -q

## Step 1: Scrape GitHub Repos

Search GitHub for repos tagged with creative-dev topics:
- `awwwards`, `awwwards-inspired` - Award-winning web design recreations
- `gsap-scrolltrigger`, `gsap-animation` - GreenSock animation platform
- `threejs` - Three.js 3D experiences
- `lenis-scroll` - Smooth scrolling library
- `animated-website` - General creative websites

Filter by >10 stars for quality signal. Extract JS/TS/CSS files containing creative patterns.

In [ ]:
# Cell 2: Run GitHub scraper
# Set your GitHub token for 5000 req/hr (vs 60 without)
import os

GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')  # Set in Colab secrets
MAX_REPOS = 500  # Adjust for testing (start with 50)
GITHUB_OUTPUT = 'data/raw/github.jsonl'

os.makedirs('data/raw', exist_ok=True)

if GITHUB_TOKEN:
    !python scripts/github_scraper.py --github-token {GITHUB_TOKEN} --output {GITHUB_OUTPUT} --max-repos {MAX_REPOS}
else:
    print('WARNING: No GITHUB_TOKEN set. Using unauthenticated API (60 req/hr).')
    print('Set GITHUB_TOKEN in Colab Secrets for 5000 req/hr.')
    !python scripts/github_scraper.py --output {GITHUB_OUTPUT} --max-repos {MAX_REPOS}

## Step 2: Scrape CodePen

Search CodePen for pens tagged with:
- `gsap`, `scrolltrigger` - GSAP animations
- `threejs`, `webgl` - 3D and WebGL experiments
- `creative-coding` - General creative code
- `lenis`, `shader` - Smooth scroll and shader art

Rate-limited to 1 request/second to respect CodePen's servers.

In [ ]:
# Cell 3: Run CodePen scraper
CODEPEN_OUTPUT = 'data/raw/codepen.jsonl'
MAX_PENS = 1000  # Adjust for testing (start with 100)

!python scripts/codepen_scraper.py --output {CODEPEN_OUTPUT} --max-pens {MAX_PENS}

## Step 3: Combine and Deduplicate Raw Data

Merge GitHub and CodePen data into a single file, removing duplicates by content hash.
This is the master raw dataset from which all training formats are derived.

In [ ]:
# Cell 4: Combine raw data, deduplicate by content hash
import json
import hashlib

def content_hash(content: str) -> str:
    """Hash content for dedup after normalizing whitespace."""
    import re
    normalized = re.sub(r'\s+', ' ', content.strip())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()[:16]

seen_hashes = set()
combined = []

for source_file in ['data/raw/github.jsonl', 'data/raw/codepen.jsonl']:
    if not os.path.exists(source_file):
        print(f'Skipping {source_file} (not found)')
        continue
    with open(source_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            chash = content_hash(entry.get('content', ''))
            if chash not in seen_hashes:
                seen_hashes.add(chash)
                combined.append(entry)

# Save combined
COMBINED_PATH = 'data/raw/combined.jsonl'
with open(COMBINED_PATH, 'w', encoding='utf-8') as f:
    for entry in combined:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f'Combined dataset: {len(combined)} unique entries')
print(f'Deduped {len(seen_hashes) - len(combined)} duplicates' if len(seen_hashes) > len(combined) else 'No duplicates found')

## Step 4: Generate SFT Dataset

Convert raw entries into chat-template instruction-response pairs.
Quality filter: only include entries with slop_score < 3.0 (genuinely creative code).

Target: 2000+ examples with full component code (50-200+ lines each).

In [ ]:
# Cell 5: Generate SFT dataset with quality filtering
import sys
sys.path.insert(0, '.')

from scripts.data_formatter import format_sft
from scripts.slop_detector import slop_score

SFT_OUTPUT = 'data/prompts/sft_prompts.jsonl'
os.makedirs('data/prompts', exist_ok=True)

sft_entries = []
skipped_quality = 0
skipped_format = 0

for entry in combined:
    content = entry.get('content', '')
    
    # Quality gate: slop_score must be < 3.0 for SFT data
    result = slop_score(content)
    if result['score'] >= 3.0:
        skipped_quality += 1
        continue
    
    # Format to chat template
    formatted = format_sft(entry)
    if formatted is None:
        skipped_format += 1
        continue
    
    sft_entries.append(formatted)

# Write SFT dataset
with open(SFT_OUTPUT, 'w', encoding='utf-8') as f:
    for entry in sft_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f'SFT dataset: {len(sft_entries)} entries')
print(f'Skipped (quality): {skipped_quality}')
print(f'Skipped (format/length): {skipped_format}')
print(f'Target: 2000+ | Status: {"MET" if len(sft_entries) >= 2000 else "NEEDS MORE DATA"}')

## Step 5: Generate SimPO Dataset

For each top-quality SFT entry:
1. Use it as the "chosen" example (hand-crafted creative code)
2. Generate a "rejected" version via rejected_generator (competent but generic)
3. Verify rejected scores as slop (is_slop=True)

This teaches the model to prefer creative, Awwwards-quality code over template garbage.

Target: 1000+ preference pairs.

In [ ]:
# Cell 6: Generate SimPO preference pairs
import re as _re
from scripts.data_formatter import format_simpo
from scripts.rejected_generator import generate_rejected

SIMPO_OUTPUT = 'data/prompts/simpo_prompts.jsonl'

simpo_entries = []
rejected_failed = 0

for entry in combined:
    content = entry.get('content', '')
    
    # Only use top-quality entries as chosen (slop_score < 2.0)
    result = slop_score(content)
    if result['score'] >= 2.0:
        continue
    
    # Generate rejected version
    # Use the detected instruction as the prompt
    from scripts.data_formatter import detect_libraries, detect_techniques, generate_instruction
    libs = detect_libraries(content)
    techs = detect_techniques(content)
    prompt = generate_instruction(content, libs, techs)
    
    rejected_code = generate_rejected(prompt, content)
    
    # Verify rejected is actually slop
    rejected_result = slop_score(rejected_code)
    if not rejected_result['is_slop']:
        rejected_failed += 1
        continue
    
    # Format as SimPO pair
    formatted = format_simpo(entry, rejected_code)
    if formatted:
        simpo_entries.append(formatted)

# Write SimPO dataset
with open(SIMPO_OUTPUT, 'w', encoding='utf-8') as f:
    for entry in simpo_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f'SimPO dataset: {len(simpo_entries)} preference pairs')
print(f'Rejected verification failures: {rejected_failed}')
print(f'Target: 1000+ | Status: {"MET" if len(simpo_entries) >= 1000 else "NEEDS MORE DATA"}')

## Step 6: Generate GRPO Prompt-Only Dataset

Extract prompts from the SFT dataset for GRPO training.
In GRPO, the model generates responses and a reward function scores them.
The `creative_code_reward` function from slop_detector.py is used as the reward.

Target: 500+ diverse prompts.

In [ ]:
# Cell 7: Generate GRPO prompt-only dataset
from scripts.data_formatter import format_grpo

GRPO_OUTPUT = 'data/prompts/grpo_prompts.jsonl'

grpo_entries = []
seen_prompts = set()

for entry in combined:
    formatted = format_grpo(entry)
    if formatted is None:
        continue
    
    # Deduplicate prompts (same instruction = redundant for GRPO)
    prompt = formatted['prompt']
    if prompt in seen_prompts:
        continue
    seen_prompts.add(prompt)
    
    grpo_entries.append(formatted)

# Write GRPO dataset
with open(GRPO_OUTPUT, 'w', encoding='utf-8') as f:
    for entry in grpo_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f'GRPO dataset: {len(grpo_entries)} unique prompts')
print(f'Target: 500+ | Status: {"MET" if len(grpo_entries) >= 500 else "NEEDS MORE DATA"}')

## Step 7: Generate KTO Binary-Label Dataset

KTO uses binary feedback signals (thumbs up / thumbs down):
- **label=True**: Entries with slop_score < 2.0 (genuinely creative code)
- **label=False**: Entries with slop_score > 6.0 (clearly generic/sloppy code)

Entries in the middle range (2.0-6.0) are excluded to give the model clear signal.

Target: 500+ binary-label entries.

In [ ]:
# Cell 8: Generate KTO binary-label dataset
from scripts.data_formatter import format_kto

KTO_OUTPUT = 'data/prompts/kto_prompts.jsonl'

kto_entries = []
kto_positive = 0
kto_negative = 0

for entry in combined:
    content = entry.get('content', '')
    result = slop_score(content)
    
    if result['score'] < 2.0:
        # Thumbs up: genuinely creative code
        formatted = format_kto(entry, label=True)
        if formatted:
            kto_entries.append(formatted)
            kto_positive += 1
    elif result['score'] > 6.0:
        # Thumbs down: clearly generic/sloppy
        formatted = format_kto(entry, label=False)
        if formatted:
            kto_entries.append(formatted)
            kto_negative += 1
    # Skip entries in the 2.0-6.0 range (ambiguous signal)

# Write KTO dataset
with open(KTO_OUTPUT, 'w', encoding='utf-8') as f:
    for entry in kto_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f'KTO dataset: {len(kto_entries)} entries')
print(f'  Positive (creative): {kto_positive}')
print(f'  Negative (generic): {kto_negative}')
print(f'Target: 500+ | Status: {"MET" if len(kto_entries) >= 500 else "NEEDS MORE DATA"}')

## Step 8: Dataset Statistics

Print comprehensive statistics about all generated datasets:
- Entry counts per dataset
- Average code length
- Quality score distributions
- Library/technique coverage

In [ ]:
# Cell 9: Print dataset statistics
print('=' * 60)
print('CIPHER CODE KRAKEN - TRAINING DATA STATISTICS')
print('=' * 60)
print()

datasets = {
    'SFT': {'path': SFT_OUTPUT, 'target': 2000, 'count': len(sft_entries)},
    'SimPO': {'path': SIMPO_OUTPUT, 'target': 1000, 'count': len(simpo_entries)},
    'GRPO': {'path': GRPO_OUTPUT, 'target': 500, 'count': len(grpo_entries)},
    'KTO': {'path': KTO_OUTPUT, 'target': 500, 'count': len(kto_entries)},
}

print(f'{"Dataset":<10} {"Count":>8} {"Target":>8} {"Status":>10}')
print('-' * 40)
for name, info in datasets.items():
    status = 'MET' if info['count'] >= info['target'] else 'LOW'
    print(f"{name:<10} {info['count']:>8} {info['target']:>8} {status:>10}")

print()
print('Raw data source breakdown:')
github_count = sum(1 for e in combined if e.get('source') == 'github')
codepen_count = sum(1 for e in combined if e.get('source') == 'codepen')
print(f'  GitHub repos: {github_count}')
print(f'  CodePen pens: {codepen_count}')
print(f'  Total unique: {len(combined)}')

# Quality score distribution
print()
print('Quality score distribution (raw data):')
score_buckets = {'< 0 (excellent)': 0, '0-2 (good)': 0, '2-5 (mixed)': 0, '> 5 (slop)': 0}
for entry in combined:
    result = slop_score(entry.get('content', ''))
    s = result['score']
    if s < 0:
        score_buckets['< 0 (excellent)'] += 1
    elif s <= 2:
        score_buckets['0-2 (good)'] += 1
    elif s <= 5:
        score_buckets['2-5 (mixed)'] += 1
    else:
        score_buckets['> 5 (slop)'] += 1

for bucket, count in score_buckets.items():
    pct = (count / len(combined) * 100) if combined else 0
    print(f'  {bucket}: {count} ({pct:.1f}%)')

print()
print('Average content length (lines):')
lengths = [entry.get('content', '').count('\n') for entry in combined]
if lengths:
    print(f'  Mean: {sum(lengths) / len(lengths):.0f} lines')
    print(f'  Min: {min(lengths)} lines')
    print(f'  Max: {max(lengths)} lines')

## Step 9: Save All Datasets

Final verification and save. All 4 datasets are written to `data/prompts/`:
- `sft_prompts.jsonl` - SFT chat-template pairs
- `simpo_prompts.jsonl` - SimPO preference pairs
- `grpo_prompts.jsonl` - GRPO prompt-only
- `kto_prompts.jsonl` - KTO binary-label triples

These are ready to be loaded by the training notebooks in Phase 1.

In [ ]:
# Cell 10: Final verification and save confirmation
import os

print('Final dataset verification:')
print()

all_outputs = {
    'SFT': SFT_OUTPUT,
    'SimPO': SIMPO_OUTPUT,
    'GRPO': GRPO_OUTPUT,
    'KTO': KTO_OUTPUT,
}

all_ok = True
for name, path in all_outputs.items():
    exists = os.path.exists(path)
    if exists:
        size = os.path.getsize(path)
        with open(path, 'r') as f:
            count = sum(1 for line in f if line.strip())
        print(f'  {name}: {path} ({count} entries, {size/1024:.1f} KB)')
    else:
        print(f'  {name}: MISSING - {path}')
        all_ok = False

print()
if all_ok:
    print('All datasets saved successfully!')
    print('Ready for training pipeline (Phase 1, Plans 02-05).')
else:
    print('WARNING: Some datasets are missing. Check scraper outputs above.')

# Verify SFT format structure
print()
print('SFT format verification (first entry):')
if os.path.exists(SFT_OUTPUT):
    with open(SFT_OUTPUT, 'r') as f:
        first = json.loads(f.readline())
    print(f'  Keys: {list(first.keys())}')
    if 'conversations' in first:
        print(f'  Conversation turns: {len(first["conversations"])}')
        for turn in first['conversations']:
            print(f'    - role: {turn["role"]}, content length: {len(turn["content"])} chars')

# Verify SimPO format structure
print()
print('SimPO format verification (first entry):')
if os.path.exists(SIMPO_OUTPUT):
    with open(SIMPO_OUTPUT, 'r') as f:
        first = json.loads(f.readline())
    print(f'  Keys: {list(first.keys())}')
    for key in ['prompt', 'chosen', 'rejected']:
        if key in first:
            print(f'    - {key}: {len(first[key])} chars')